# pandas: aggregation

Aggregation is fundamental to understanding the contours of a data set.

In this lesson I'll discuss various `DataFrame` aggregation methods including group-wise operations as we continue to explore the Mega Millions lottery data covering the period 2017-2024.

💡 I've adjusted the `IPython` interactive shell's `ast_node_interactivity` setting to ensure that all cell output is displayed. This overrides the default last expression only ("last_expr") display behavior.

In [1]:
import altair as alt
import numpy as np
import IPython as ipy
import pandas as pd


# Ensure that all interactive cell output is displayed (default="last_expr")
ipy.get_ipython().ast_node_interactivity = "all"

## Retrieve and prep the data

Let's read in the winning numbers data, convert the "draw_date" column dtype from `object` to `datetime64[ns]`, review a summary of the new `DataFrame` named `data`, and inspect its first three rows.

In [2]:
data = pd.read_csv("./data/mega_millions_pick5_cols-2017_24.csv")
data["draw_date"] = pd.to_datetime(data["draw_date"], format="%Y-%m-%d")
data.info()
data.head(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 684 entries, 0 to 683
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   draw_date        684 non-null    datetime64[ns]
 1   winning_numbers  684 non-null    object        
 2   mega_ball        684 non-null    int64         
 3   multiplier       684 non-null    int64         
 4   pick5_1          684 non-null    int64         
 5   pick5_2          684 non-null    int64         
 6   pick5_3          684 non-null    int64         
 7   pick5_4          684 non-null    int64         
 8   pick5_5          684 non-null    int64         
dtypes: datetime64[ns](1), int64(7), object(1)
memory usage: 48.2+ KB


,draw_date,winning_numbers,mega_ball,multiplier,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5
0,2024-05-17,08 17 40 60 70,3,2,8,17,40,60,70
1,2024-05-14,13 19 43 62 64,6,3,13,19,43,62,64
2,2024-05-10,13 22 26 32 65,18,4,13,22,26,32,65


## Aggregation

Both the pandas `Series` and `DataFrame` are provisioned with methods for computing aggregations such as the mean, median, min, max, standard deviation, and sum. Let's begin our exploration of how to compute summary statistics by exploring the Mega Millions Pick 5 winning number draws.

### Mega Millions "Pick 5" selection strategies

When purchasing a \$2.00 Mega Millions lottery ticket a player selects five "Pick 5" numbers between `1` and `70` and a single "Mega Ball" number between `1` and `25`. For an additional $1.00 per ticket, the player can add a "Megaplier" to increase non-jackpot winnings by 2x, 3x, 4x, or 5x.

Let's see if we can determine whether or not a player should consider a "Pick 5" selection strategy that opts for all low numbers, all high numbers, or a mix of low and high numbers based on previous winning draws.

One approach is to compute the row-wise sums of the Pick 5 columns and then count the number of times each sum appears among the results. Why do this? The low sum/high sum frequency counts should allow us to test whether or not an all low number or all high number selection strategy is a viable option.

To sum the values across the "pick5_*" columns call the [`DataFrame.sum()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sum.html) method and pass it the keyword argument `axis=1` to ensure a row-wise summing operation. I'll assign each sum returned to a new column named "pick5_sum".

In [3]:
data.loc[:, "pick5_sum"] = data.loc[:, "pick5_1":"pick5_5"].sum(axis=1)
data.head()

,draw_date,winning_numbers,mega_ball,multiplier,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5,pick5_sum
0,2024-05-17,08 17 40 60 70,3,2,8,17,40,60,70,195
1,2024-05-14,13 19 43 62 64,6,3,13,19,43,62,64,201
2,2024-05-10,13 22 26 32 65,18,4,13,22,26,32,65,158
3,2024-05-07,26 28 36 63 66,15,3,26,28,36,63,66,219
4,2024-05-03,06 13 15 53 56,11,2,6,13,15,53,56,143


### Compute the mean

You can compute the mean of the row-wise summing operation by calling the [`DataFrame.mean()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.mean.html) method.

In [4]:
data.loc[:, "pick5_sum"].mean()

173.23976608187135

### Aggregate summary operations with `DataFrame.aggregate()`

Better yet, you can compute various summary statistics by passing a sequence of method names to the [`DataFrame.aggregate()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.aggregate.html) method, which is also aliased as [`DataFrame.agg()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.agg.html).

In [5]:
data.loc[:, "pick5_sum"].agg(["mean", "median", "min", "max", "std"])
# data.loc[:, "pick5_sum"].aggregate(["mean", "median", "min", "max", "std"])  # Alternative

mean      173.239766
median    173.000000
min        48.000000
max       281.000000
std        43.945639
Name: pick5_sum, dtype: float64

### Plot it

Let's employ a bar chart to visualize the frequency counts of the Pick 5 summing operations.

💡 This notebook features a number of [Vega-Altair](https://altair-viz.github.io/) bar charts. I've defined a function named `encode_mark_bar()` that standardizes the encoding of each plot. Implementing functions eliminates code duplication and enhances code modularization.

In [6]:
def encode_mark_bar(
    base,
    x_shorthand,
    x_axis_title,
    x_maxbins=15,
    x_scale_domain=(0, 305),
    y_shorthand="count():Q",
    y_axis_title="Count",
    y_scale_domain=(0, 30),
    color_shorthand="count():Q",
    color_title="Count",
    color_scale_scheme="blues",
):
    """Encode a bar chart with text annotations.

    Parameters:
        base (alt.Chart): The base chart to encode.
        x_shorthand (str): The x-axis shorthand (field, aggregate, type).
        x_axis_title (str): The x-axis title.
        x_maxbins (int): The maximum number of bins.
        x_scale_domain (tuple): The x-axis scale domain.
        y_shorthand (str): The y-axis shorthand (field, aggregate, type).
        y_axis_title (str): The y-axis title.
        y_scale_domain (tuple): The y-axis scale domain.
        color_shorthand (str): The color shorthand (field, aggregate, type).
        color_title (str): The color title.
        color_scale_scheme (str): The color scheme for the chart.

    Returns:
        alt.LayerChart: The encoded chart.
    """

    bar = base.mark_bar().encode(
        x=alt.X(
            shorthand=x_shorthand,
            axis=alt.Axis(
                labelAngle=-60,
                labelFontWeight="normal",
                tickCount=15,
                tickMinStep=5,
                title=x_axis_title,
                titleFontSize=10,
                titleFontWeight="normal",
            ),
            bin=alt.Bin(maxbins=x_maxbins),
            scale=alt.Scale(domain=x_scale_domain, padding=0.3),
        ),
        y=alt.Y(
            shorthand=y_shorthand,
            axis=alt.Axis(
                labelFontWeight="normal",
                title=y_axis_title,
                titleFontSize=10,
                titleFontWeight="normal",
            ),
            scale=alt.Scale(domain=y_scale_domain),
        ),
        color=alt.Color(
            shorthand=color_shorthand,
            legend=alt.Legend(
                gradientThickness=7,
                title=color_title,
                titleFontSize=10,
                titleFontWeight="bold",
                tickCount=5,
                tickMinStep=5,
                labelFontSize=8,
                labelFontWeight="normal",
            ),
            scale=alt.Scale(scheme=color_scale_scheme),
        ),
    )

    return bar

The code below instantiates an instance of an `alt.Chart()` object, calls the function `encode_mark_bar()` to encode the bar chart, and then adds additional encodings and labels to the chart. What does the chart suggest about the distribution of Pick 5 winning number sums and number selection strategies?

In [7]:
base = alt.Chart(data)
bar = encode_mark_bar(
    base, "pick5_sum:Q", "Pick 5 (sum)", x_scale_domain=(0, 320), y_scale_domain=(0, 140)
)
text = bar.mark_text(align="center", baseline="bottom", fontSize=8).encode(
    text="count():Q", color=alt.value("black")
)
rule = base.mark_rule().encode(
    alt.X("median(pick5_sum):Q"),
    color=alt.value("red"),
    strokeWidth=alt.value(2),
)
chart = (bar + text + rule).properties(
    title=alt.Title("Pick 5 winning number sums", align="center"), width=300, height=300
)
chart.show()

alt.LayerChart(...)

## Group-wise aggregation

You can also separate data into groups and compute summary statistics for each group. Both the `DataFrame` and `Series` are provisioned with a `groupby()` method that groups data by a "mapper" object (e.g., label, `list`, `dict`, `Series`, function). You can split the data along either axis (i.e., index or columns), sort the resulting groups, and drop groups containing missing values. The method call returns an instance of the [`DataFrameGroupBy`](https://pandas.pydata.org/docs/reference/groupby.html) class, an [_iterable_](https://docs.python.org/3/glossary.html#term-iterable) that references each group created by the operation.

`DataFrame.groupby(by=None, axis=_NoDefault.no_default, level=None, as_index=True, sort=True, group_keys=True, observed=_NoDefault.no_default, dropna=True)`

### Group by year

Let's group the winning draws by year. First, I'll create a "mapper" by accessing the year segment of each "draw_date" value in `data` and assign the resulting `Series` to the variable named `mapper`. Next, I'll call the `DataFrame.groupby()` method, pass it `mapper` and the keyword argument `sort=False` to ensure the original order is preserved.

In [8]:
mapper = data["draw_date"].dt.year  # Series
data_years = data.groupby(mapper, sort=False)  # preserve original order

### Size

You can return the size of each group by calling the [`DataFrameGroupBy.size()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.size.html) method.

In [9]:
data_years.size()

draw_date
2024     40
2023    104
2022    104
2021    105
2020    104
2019    105
2018    104
2017     18
dtype: int64

### First and Last

You can return the first or last row in each group by calling either the
[`DataFrameGroupBy.first()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.first.html) or [`DataFrameGroupBy.last()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.last.html) methods.

In [10]:
data_years.first()
data_years.last()

,draw_date,winning_numbers,mega_ball,multiplier,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5,pick5_sum
draw_date,,,,,,,,,,
2024,2024-05-17,08 17 40 60 70,3,2,8,17,40,60,70,195
2023,2023-12-29,11 27 30 62 70,10,3,11,27,30,62,70,200
2022,2022-12-30,01 03 06 44 51,7,3,1,3,6,44,51,105
2021,2021-12-31,02 05 30 46 61,8,3,2,5,30,46,61,144
2020,2020-12-29,01 31 35 48 62,19,3,1,31,35,48,62,177
2019,2019-12-31,30 44 49 53 56,11,3,30,44,49,53,56,232
2018,2018-12-28,09 10 25 37 38,21,2,9,10,25,37,38,119
2017,2017-12-29,04 10 18 28 62,7,2,4,10,18,28,62,122


,draw_date,winning_numbers,mega_ball,multiplier,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5,pick5_sum
draw_date,,,,,,,,,,
2024,2024-01-02,03 18 27 29 64,1,2,3,18,27,29,64,141
2023,2023-01-03,25 29 33 41 44,18,4,25,29,33,41,44,172
2022,2022-01-04,04 06 16 21 22,1,3,4,6,16,21,22,69
2021,2021-01-01,08 24 53 68 69,7,5,8,24,53,68,69,222
2020,2020-01-03,37 41 42 53 63,16,2,37,41,42,53,63,236
2019,2019-01-01,34 44 57 62 70,14,4,34,44,57,62,70,267
2018,2018-01-02,01 42 47 64 70,22,4,1,42,47,64,70,224
2017,2017-10-31,06 28 31 52 53,12,4,6,28,31,52,53,170


### Access a group

Call the [`DataFrameGroupBy.get_group()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.get_group.html) method to access a specific group.

In [11]:
data_years.get_group(2021)

,draw_date,winning_numbers,mega_ball,multiplier,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5,pick5_sum
248,2021-12-31,02 05 30 46 61,8,3,2,5,30,46,61,144
249,2021-12-28,03 05 08 31 38,4,3,3,5,8,31,38,85
250,2021-12-24,16 17 25 36 37,16,2,16,17,25,36,37,131
251,2021-12-21,25 31 58 64 67,24,3,25,31,58,64,67,245
252,2021-12-17,21 32 38 48 62,10,3,21,32,38,48,62,201
...,...,...,...,...,...,...,...,...,...,...
348,2021-01-15,03 11 12 38 43,15,4,3,11,12,38,43,107
349,2021-01-12,12 14 26 28 33,9,2,12,14,26,28,33,113
350,2021-01-08,03 06 16 18 58,11,2,3,6,16,18,58,101
351,2021-01-05,20 43 51 55 57,4,2,20,43,51,55,57,226


### Binning

If you need to specify discrete intervals for each desired group, utilize the `pd.cut()` function in tandem with the `DataFrame.groupBy()` method.

`pandas.cut(x, bins, right=True, labels=None, retbins=False, precision=3, include_lowest=False, duplicates='raise', ordered=True)`

In the following example, I employ `pd.cut()` along with `np.arange()` to provide the bin intervals upon which to group the data by the "pick5_sum" column. Passed directly to `DataFrame.groupby()` the expression orchestrates the grouping operation resulting in a `DataFrameGroupBy` object containing ten (`10`) groups.

You can return the size of each group by calling the [`DataFrameGroupBy.size()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.size.html) method.

In [12]:
data_bins = data.groupby(
    pd.cut(data["pick5_sum"], bins=np.arange(15, 341, 30)), observed=False
)  # pass observed=False in response to a DeprecationWarning
data_bins.size()

pick5_sum
(15, 45]        0
(45, 75]       11
(75, 105]      34
(105, 135]     93
(135, 165]    159
(165, 195]    160
(195, 225]    149
(225, 255]     59
(255, 285]     19
(285, 315]      0
dtype: int64

Calling the [`DataFrameGroupBy.max()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.max.html) method returns the row with the maximum Pick 5 sum value in each group; calling the[`DataFrameGroupBy.min()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.min.html) method returns the row with the minimum Pick 5 sum value in each group.

In [13]:
data_bins.max()
data_bins.min()

,draw_date,winning_numbers,mega_ball,multiplier,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5,pick5_sum
pick5_sum,,,,,,,,,,
"(15, 45]",NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"(45, 75]",2024-01-19,07 13 14 15 18,25.0,4.0,7.0,13.0,16.0,21.0,38.0,75.0
"(75, 105]",2023-01-31,11 16 23 24 30,25.0,5.0,11.0,16.0,26.0,44.0,70.0,105.0
"(105, 135]",2024-04-12,17 22 25 30 38,25.0,5.0,17.0,24.0,33.0,52.0,70.0,135.0
"(135, 165]",2024-05-10,23 24 35 40 43,25.0,5.0,23.0,31.0,46.0,63.0,70.0,165.0
"(165, 195]",2024-05-17,28 31 41 42 50,25.0,5.0,28.0,38.0,56.0,67.0,70.0,195.0
"(195, 225]",2024-05-14,34 35 41 45 54,25.0,5.0,34.0,47.0,61.0,69.0,70.0,225.0
"(225, 255]",2024-04-09,39 45 52 56 59,25.0,5.0,39.0,54.0,62.0,69.0,70.0,253.0
"(255, 285]",2023-06-23,46 54 57 58 66,25.0,5.0,46.0,62.0,66.0,69.0,70.0,281.0


,draw_date,winning_numbers,mega_ball,multiplier,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5,pick5_sum
pick5_sum,,,,,,,,,,
"(15, 45]",NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"(45, 75]",2018-05-04,01 02 04 19 29,1.0,2.0,1.0,2.0,4.0,10.0,17.0,48.0
"(75, 105]",2017-11-14,01 03 05 08 70,1.0,2.0,1.0,3.0,5.0,8.0,25.0,76.0
"(105, 135]",2017-11-21,01 02 11 52 64,1.0,2.0,1.0,2.0,6.0,17.0,27.0,106.0
"(135, 165]",2017-12-15,01 07 40 43 68,1.0,2.0,1.0,5.0,6.0,28.0,42.0,136.0
"(165, 195]",2017-10-31,01 04 50 54 59,1.0,2.0,1.0,4.0,20.0,35.0,44.0,166.0
"(195, 225]",2017-11-03,01 28 61 62 63,1.0,2.0,1.0,11.0,27.0,41.0,47.0,196.0
"(225, 255]",2017-11-07,01 54 60 68 69,1.0,2.0,1.0,23.0,39.0,45.0,55.0,226.0
"(255, 285]",2018-07-17,11 59 66 67 68,1.0,2.0,11.0,36.0,51.0,57.0,62.0,258.0


You can ignore non-numeric columns by passing the keyword argument `numeric_only=True` to the `max()` and `min()` methods.

In [14]:
data_bins.max(numeric_only=True)
data_bins.min(numeric_only=True)

,mega_ball,multiplier,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5,pick5_sum
pick5_sum,,,,,,,,
"(15, 45]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"(45, 75]",25.0,4.0,7.0,13.0,16.0,21.0,38.0,75.0
"(75, 105]",25.0,5.0,11.0,16.0,26.0,44.0,70.0,105.0
"(105, 135]",25.0,5.0,17.0,24.0,33.0,52.0,70.0,135.0
"(135, 165]",25.0,5.0,23.0,31.0,46.0,63.0,70.0,165.0
"(165, 195]",25.0,5.0,28.0,38.0,56.0,67.0,70.0,195.0
"(195, 225]",25.0,5.0,34.0,47.0,61.0,69.0,70.0,225.0
"(225, 255]",25.0,5.0,39.0,54.0,62.0,69.0,70.0,253.0
"(255, 285]",25.0,5.0,46.0,62.0,66.0,69.0,70.0,281.0


,mega_ball,multiplier,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5,pick5_sum
pick5_sum,,,,,,,,
"(15, 45]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"(45, 75]",1.0,2.0,1.0,2.0,4.0,10.0,17.0,48.0
"(75, 105]",1.0,2.0,1.0,3.0,5.0,8.0,25.0,76.0
"(105, 135]",1.0,2.0,1.0,2.0,6.0,17.0,27.0,106.0
"(135, 165]",1.0,2.0,1.0,5.0,6.0,28.0,42.0,136.0
"(165, 195]",1.0,2.0,1.0,4.0,20.0,35.0,44.0,166.0
"(195, 225]",1.0,2.0,1.0,11.0,27.0,41.0,47.0,196.0
"(225, 255]",1.0,2.0,1.0,23.0,39.0,45.0,55.0,226.0
"(255, 285]",1.0,2.0,11.0,36.0,51.0,57.0,62.0,258.0


You can also leverage the [`DataFrameGroupBy.aggregate()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.aggregate.html) method (also aliased as [`agg()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.agg.html)) to compute multiple aggregations across a specified axis. For example I can compute the minimum and maximum values in `data_bins` by passing a sequence of aggregate labels to the method.

In [15]:
data_bins.aggregate(["min", "max"])
# data_bins.agg(["min", "max"])  # Alternative

draw_date            winning_numbers                 mega_ball  \
                  min        max             min             max       min   
pick5_sum                                                                    
(15, 45]          NaT        NaT             NaN             NaN       NaN   
(45, 75]   2018-05-04 2024-01-19  01 02 04 19 29  07 13 14 15 18       1.0   
(75, 105]  2017-11-14 2023-01-31  01 03 05 08 70  11 16 23 24 30       1.0   
(105, 135] 2017-11-21 2024-04-12  01 02 11 52 64  17 22 25 30 38       1.0   
(135, 165] 2017-12-15 2024-05-10  01 07 40 43 68  23 24 35 40 43       1.0   
(165, 195] 2017-10-31 2024-05-17  01 04 50 54 59  28 31 41 42 50       1.0   
(195, 225] 2017-11-03 2024-05-14  01 28 61 62 63  34 35 41 45 54       1.0   
(225, 255] 2017-11-07 2024-04-09  01 54 60 68 69  39 45 52 56 59       1.0   
(255, 285] 2018-07-17 2023-06-23  11 59 66 67 68  46 54 57 58 66       1.0   
(285, 315]        NaT        NaT             NaN             NaN       NaN   

                 multiplier      pick5_1       pick5_2       pick5_3        \
             max        min  max     min   max     min   max     min   max   
pick5_sum                                                                    
(15, 45]     NaN        NaN  NaN     NaN   NaN     NaN   NaN     NaN   NaN   
(45, 75]    25.0        2.0  4.0     1.0   7.0     2.0  13.0     4.0  16.0   
(75, 105]   25.0        2.0  5.0     1.0  11.0     3.0  16.0     5.0  26.0   
(105, 135]  25.0        2.0  5.0     1.0  17.0     2.0  24.0     6.0  33.0   
(135, 165]  25.0        2.0  5.0     1.0  23.0     5.0  31.0     6.0  46.0   
(165, 195]  25.0        2.0  5.0     1.0  28.0     4.0  38.0    20.0  56.0   
(195, 225]  25.0        2.0  5.0     1.0  34.0    11.0  47.0    27.0  61.0   
(225, 255]  25.0        2.0  5.0     1.0  39.0    23.0  54.0    39.0  62.0   
(255, 285]  25.0        2.0  5.0    11.0  46.0    36.0  62.0    51.0  66.0   
(285, 315]   NaN        NaN  NaN     NaN   NaN     NaN   NaN     NaN   NaN   

           pick5_4       pick5_5       pick5_sum         
               min   max     min   max       min    max  
pick5_sum                                                
(15, 45]       NaN   NaN     NaN   NaN       NaN    NaN  
(45, 75]      10.0  21.0    17.0  38.0      48.0   75.0  
(75, 105]      8.0  44.0    25.0  70.0      76.0  105.0  
(105, 135]    17.0  52.0    27.0  70.0     106.0  135.0  
(135, 165]    28.0  63.0    42.0  70.0     136.0  165.0  
(165, 195]    35.0  67.0    44.0  70.0     166.0  195.0  
(195, 225]    41.0  69.0    47.0  70.0     196.0  225.0  
(225, 255]    45.0  69.0    55.0  70.0     226.0  253.0  
(255, 285]    57.0  69.0    62.0  70.0     258.0  281.0  
(285, 315]     NaN   NaN     NaN   NaN       NaN    NaN

When appropriate you can also compute the [mean](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.mean.html), [median](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.median.html), [standard deviation](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.std.html), [sum](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.sum.html), and other [descriptive statistics](https://pandas.pydata.org/docs/reference/groupby.html) for each group.

## Grouping on multiple columns

You can also group on multiple columns by passing a sequence of column keys to the `DataFrame.groupby()` method.

## Even numbers, odd numbers, or a mix of both

A friend who occasionally plays the Mega Millions lottery wondered if the _parity_ of each of the Mega Ball and Pick 5 numbers&mdash;i.e., whether the number is **even** or **odd**&mdash;influences the game in any way. More particularly, does selecting all even numbers, or all odd numbers, or a particular mix of each increase the odds of winning?

Interesting question and one that suggests that we will need to group the Mega Ball and Pick 5 winning draws by the parity of each selected number (i.e., all even Mega Ball/Pick 5 numbers, all odd Mega Ball/Pick 5 numbers, etc.).

### Parity groups

We can determine the parity of each winning number by evaluating whether or not the number is divisible evenly by `2`. For example, `12` can be divided evenly by `2` (6 times) while `31` cannot. You can call the [`DataFrame.mod()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.mod.html) method and pass it a modulus of `2` to check if a number is even or odd. If the modulo operation resolves to `0` the number is **even**; if it resolves to `1` the number is **odd**.

`DataFrame.mod(other, axis='columns', level=None, fill_value=None)`

The code below computes the parity of each Mega Ball winning number and stores the result of the modulo operation in a new `data` column named "mega_ball_mod2" inserted into `data` after the "mega_ball" column.

In [16]:
mega_ball_mod2 = data.loc[:, "mega_ball"].mod(2)
data.insert(data.columns.get_loc("mega_ball") + 1, "mega_ball_mod2", mega_ball_mod2)
data["mega_ball_mod2"].unique()  # Confirm values

array([1, 0])

Next, the "Pick 5" columns are passed to `DataFrame.mod(2)`. By default, the method operates row-wise and returns either `0` or `1` for each of the five winning numbers encountered. The sum of each of the five modulo operations computed for each row (e.g., `39 50 61 64 70` -> `1 0 1 0 0` = `2`) is stored in a new column named "pick5_mod2_sum".

In [17]:
data["pick5_mod2_sum"] = data.loc[:, "pick5_1":"pick5_5"].mod(2).sum(axis=1)
data["pick5_mod2_sum"].unique()  # Confirm values

array([1, 3, 2, 5, 0, 4])

The `DataFrame.groupby()` method can now be invoked to split the winning draws into groups based on the "mega_ball_mod2" and "pick5_mod2_sum" columns. Doing so allows us to assess the degree to which the winning draws incline towards all even or all odd selections.

#### Mega Ball groups

* 0 = Even number selected
* 1 = Odd number selected

#### Pick 5 groups

* 0 = All even numbers selected
* 1 = Four even numbers, one odd number selected
* 2 = Three even numbers, two odd numbers selected
* 3 = Two even numbers, three odd numbers selected
* 4 = One even number, four odd numbers selected
* 5 = All odd numbers selected

In [18]:
data_groups = data.groupby(["mega_ball_mod2", "pick5_mod2_sum"])
data_groups.size()

mega_ball_mod2  pick5_mod2_sum
0               0                  12
                1                  54
                2                 118
                3                  93
                4                  45
                5                  10
1               0                  10
                1                  59
                2                 108
                3                 120
                4                  49
                5                   6
dtype: int64

### Plot it

Next, I'll generate a Vega-Altair concatenated chart based on the groups in `data_groups`. Recall that `data_groups` is an instance of the `DataFrameGroupBy` class, an _iterable_ capable of returning its members one at a time. This means that I can iterate over `data_groups`, access each `DataFrame`, and generate a plot based on it.

💡 The function `assign_parity_label()` provides a human-readable label for each plot group.

In [19]:
def assign_parity_label(remainder):
    """Based on the passed in modulo remainder, returns the appropriate "parity" label
    indicating the mix of even and/or odd numbers that consititute a player's Pick 5
    selections.

    Parameters:
        row (Series): Series of n integer elements

    Returns:
        str: label designating all even, all odd, or mixed parity
    """

    match remainder:
        case 0:
            return "all even"
        case 1:
            return "4 even, 1 odd"
        case 2:
            return "3 even, 2 odd"
        case 3:
            return "2 even, 3 odd"
        case 4:
            return "1 even, 4 odd"
        case 5:
            return "all odd"
        case _:
            return "N/A"

In the example below I loop over `data_groups`, construct a bar chart for each group, and store the chart objects in a dictionary named "plots".

In [20]:
plots = {"even": [], "odd": []}
for name, group in data_groups:
    mega_ball, pick5 = name  # unpack
    mega_ball_label = "odd" if mega_ball else "even"  # ternary
    title = f"Mega Ball ({mega_ball_label}): Pick 5 ({assign_parity_label(pick5)})"

    bar = encode_mark_bar(
        alt.Chart(group), "pick5_sum:Q", "Pick 5 (sum)", color_scale_scheme="purples"
    )
    text = bar.mark_text(align="center", baseline="bottom", fontSize=8).encode(
        text="count():Q", color=alt.value("black")
    )

    plot = (bar + text).properties(
        title=alt.Title(title, align="center", fontSize=10, fontWeight="bold"),
        height=150,
        width=150,
    )

    plots["odd"].append(plot) if mega_ball else plots["even"].append(plot)

print(f"{len(plots['even'])}, {len(plots['odd'])}")  # chart count

6, 6


After I've mapped the chart objects to the `plots` dictionary, I instantiate an instance of `alt.ConcatChart()` and concatenate the "even" and "odd" plots vertically with `alt.vconcat()`.

Taken together, the various plots suggest a binomial distribution but confirmation will need to wait for our later lessons on probability theory. In other words, hold off drawing conclusions about the optimal mix of even/odd Mega Ball and Pick5 number selections.

In [21]:
chart = alt.ConcatChart(
    concat=[alt.vconcat(*plots["even"]), alt.vconcat(*plots["odd"])],
    columns=2,
    spacing=10,
    title=alt.TitleParams(
        "Mega Ball, Pick 5 even/odd selections",
        align="center",
        anchor="middle",
        fontSize=12,
        fontWeight="bold",
    ),
)

chart.show()

alt.ConcatChart(...)

Grouping on multiple columns increases the complexity of the resulting `DataFrameGroupBy` object, yet aggregations can still be performed.

In [22]:
data_groups.agg(["min", "max"])

draw_date            winning_numbers  \
                                     min        max             min   
mega_ball_mod2 pick5_mod2_sum                                         
0              0              2018-03-02 2023-12-15  02 10 46 50 56   
               1              2017-11-10 2024-04-12  01 12 14 18 66   
               2              2017-10-31 2024-05-10  01 03 12 22 42   
               3              2017-12-05 2024-05-14  01 02 04 19 29   
               4              2018-01-30 2024-02-16  01 03 19 25 58   
               5              2018-01-16 2024-04-23  01 11 37 47 51   
1              0              2017-12-29 2024-04-02  02 14 16 38 66   
               1              2017-12-15 2024-05-17  01 10 18 20 46   
               2              2017-11-03 2024-03-08  01 02 11 52 64   
               3              2017-11-21 2024-05-03  01 02 23 40 45   
               4              2017-11-28 2024-01-12  01 21 25 27 40   
               5              2018-10-16 2024-04-26  03 45 49 61 69   

                                              mega_ball     multiplier      \
                                          max       min max        min max   
mega_ball_mod2 pick5_mod2_sum                                                
0              0               24 28 42 60 64         6  24          2   4   
               1               46 54 57 58 66         2  24          2   5   
               2               39 40 52 60 67         2  24          2   5   
               3               40 41 61 66 67         2  24          2   5   
               4               37 41 42 53 63         2  24          2   5   
               5               29 35 59 61 69         4  24          2   4   
1              0               30 32 42 46 48         1  21          2   4   
               1               29 52 58 60 62         1  25          2   5   
               2               30 44 49 53 56         1  25          2   5   
               3               39 45 52 56 59         1  25          2   5   
               4               41 43 51 57 70         1  25          2   4   
               5               33 35 41 45 51         1  25          2   5   

                              pick5_1     pick5_2     pick5_3     pick5_4      \
                                  min max     min max     min max     min max   
mega_ball_mod2 pick5_mod2_sum                                                   
0              0                    2  24       6  40      14  60      24  64   
               1                    1  46       3  54      13  58      18  66   
               2                    1  39       3  58      10  61      16  69   
               3                    1  40       2  59       4  66      17  68   
               4                    1  37       3  62       9  65      10  67   
               5                    1  29       9  35      15  59      19  61   
1              0                    2  30      10  50      16  56      24  66   
               1                    1  29       5  52      10  58      12  66   
               2                    1  30       2  54      10  62      17  69   
               3                    1  39       2  47       5  61       8  69   
               4                    1  41       5  53       9  65      13  69   
               5                    3  33      11  45      13  53      31  61   

                              pick5_5     pick5_sum       
                                  min max       min  max  
mega_ball_mod2 pick5_mod2_sum                             
0              0                   46  68       114  244  
               1                   31  70        75  281  
               2                   30  70        76  268  
               3                   27  70        55  275  
               4                   23  70        48  276  
               5                   25  69        75  253  
1              0                   46  68      

### Write to file

Let's conclude our discussion of `DataFrame` aggregation by writing `data` to a CSV file. We will make use of the data file in a future lesson. 🙂

In [23]:
filepath = "./data/mega_millions-aggregate-2017_24.csv"
data.to_csv(filepath, index=False)